Here we'll utilise Grad-CAM to visualize our model decisions boundries to understand them and observe why certain model fails at the given task.

In [6]:
import os
import struct
import torch
import torchvision
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from torchinfo import summary
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [5]:
class my_dataset(Dataset):
    def __init__(self, root: str, train: bool = True, transform = None):

        self.root = root
        self.train = train
        self.transform = transform

        self.classes = ["T-shirt/top","Trouser","Pullover","Dress","Coat","Sandal","Shirt","Sneaker","Bag","Ankle boot"]

        if train:
            image_path = "train-images-idx3-ubyte"
            label_path = "train-labels-idx1-ubyte"
        else:
            image_path = "t10k-images-idx3-ubyte"
            label_path = "t10k-labels-idx1-ubyte"

        self.img_path = os.path.join(root, image_path)
        self.lbl_path = os.path.join(root, label_path)

        self.samples = self._load_image(self.img_path)
        self.labels = self._load_label(self.lbl_path)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        sample = self.samples[index]
        label = self.labels[index]

        if self.transform is not None:
            sample = self.transform(sample)

        if self.transform is None and not self.train:
            sample = sample.unsqueeze(0)

        return sample, label

    def _load_image(self, path):
        with open(path, "rb") as f:
            magic, no_of_samples, rows, cols = struct.unpack(">IIII", f.read(16))
            datas = f.read()

            samples = torch.tensor(list(datas), dtype = torch.float32)
            samples = samples/255.0
            samples = samples.reshape(no_of_samples, rows, cols)

        return samples

    def _load_label(self, path):
        with open(path, "rb") as f:
            magic, no_of_samples = struct.unpack(">II", f.read(8))
            datas = f.read()

            labels = torch.tensor(list(datas), dtype = torch.long)

        return labels

In [7]:
train_transform = torchvision.transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandAugment(num_ops = 2, magnitude = 9, num_magnitude_bins = 31),
    transforms.ToTensor()
])

In [8]:
root = Path(r"data/FashionMNIST/raw")

train_data = my_dataset(root, train = True, transform = train_transform)
test_data = my_dataset(root, train = False, transform = None)

train_sample, train_label = train_data[0]
test_sample, test_label = test_data[0]

train_sample.shape, train_data.classes[train_label.item()], test_sample.shape, test_data.classes[test_label.item()]

(torch.Size([1, 28, 28]), 'Ankle boot', torch.Size([1, 28, 28]), 'Ankle boot')

In [9]:
def my_gradcam(model: torch.nn.Module, data_sample, target, device: torch.device):
    model = model.to(device)
    img, label = data_sample
    sample = img.to(device)

    classes = ["T-shirt/top","Trouser","Pullover","Dress","Coat","Sandal","Shirt","Sneaker","Bag","Ankle boot"]

    activation_map = {}
    gradients = {}

    def forward_hook(module, input, output):
        activation_map['value'] = output

    def backward_hook(module, grad_input, grad_output):
        gradients['value'] = grad_output[0]

    def min_max_normalize(data):
        data = data - data.min()
        data = data / (data.max() + 1e-8)
        return data

    forward_handel = target.register_forward_hook(forward_hook)
    backward_handel = target.register_full_backward_hook(backward_hook)

    try:
        model.eval()
        y_pred = model(sample)
    
        pred_class_idx = y_pred.argmax(dim = 1).item()
        pred_label = classes[pred_class_idx]

        model.zero_grad()
        score = y_pred[0, pred_class_idx]
        score.backward()

        activation_map_values = activation_map['value']
        gradients_values = gradients['value']

        importance = torch.mean(gradients_values, dim = (2,3), keepdim = True)
        gradients_weighted = importance * activation_map_values

        alpha = torch.sum(gradients_weighted, dim = 1, keepdim = True)
        cam = torch.nn.functional.relu(alpha)

        upscaled_cam = torch.nn.functional.interpolate(cam, size = (28, 28), mode = 'bilinear', align_corners = False)

        normalised_cam = min_max_normalize(upscaled_cam)

        heatmap_overlay = normalised_cam.squeeze().detach().cpu().numpy()

        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        
        axes[0].imshow(sample.squeeze().detach().cpu().numpy(), cmap='gray')
        axes[0].set_title(f"Original\nTrue: {classes[label]}")
        axes[0].axis('off')
        
        axes[1].imshow(heatmap_overlay, cmap='jet')
        axes[1].set_title(f"Grad-CAM\nTarget: {pred_label}")
        axes[1].axis('off')
        
        axes[2].imshow(sample.squeeze().detach().cpu().numpy(), cmap='gray')
        axes[2].imshow(heatmap_overlay, cmap='jet', alpha=0.5)
        axes[2].set_title("Overlay")
        axes[2].axis('off')
        
        plt.tight_layout()
        plt.show()

    finally:
        forward_handel.remove()
        backward_handel.remove()